# Entrenamiento local

Cada integrante del equipo entrena el modelo global (`CaffeLeNet.py`) con su partición privada de MNIST y entrega el modelo resultante al servidor. Este cuaderno contiene únicamente el ciclo de entrenamiento y la evaluación de cada modelo local: curvas de aprendizaje y *classification report*.

En un despliegue real, cada integrante ejecuta este cuaderno en su propia computadora con `CLIENT_IDS = [k]`. Para simular el flujo completo, se pueden entrenar todos los clientes en una sola ejecución.

In [ ]:
import json
import os

import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from CaffeLeNet import build

### Configuración

- `DATA_DIR`: carpeta con las particiones privadas `client_{k}.npz` (no versionadas; ver `README.md`).
- `BATCH_SIZE = 64`: tamaño de lote del ejemplo de MNIST de Caffe.
- `VAL_SIZE`: fracción de los datos locales reservada para validación. Un cliente no tiene acceso a los datos de prueba del servidor, por lo que las curvas de aprendizaje usan validación local.
- `INIT_MODES`: se entrenan dos conjuntos de modelos locales, que el servidor usa en escenarios distintos (ver `global_training.ipynb`):
  - `lmodel`: todos los clientes parten del mismo modelo inicial $w_0$, como si el servidor lo distribuyera antes del entrenamiento. FedAvg asume esta condición (McMahan et al., 2017).
  - `lmodel_indep`: cada cliente parte de una inicialización aleatoria propia, sin coordinación con el servidor.

In [ ]:
DATA_DIR = "./private/data"
CLIENT_IDS = [0, 1, 2, 3]  # en un deploy real, únicamente el id propio
EPOCHS = 5
BATCH_SIZE = 64
VAL_SIZE = 0.1
SEED = 42
INIT_MODES = {"lmodel": True, "lmodel_indep": False} 

for folder in ("models", "reports", "viz"):
    os.makedirs(folder, exist_ok=True)

### Datos

El conjunto de prueba de MNIST es común a todos los clientes y solo se usa para el *classification report* final de cada modelo local.

In [ ]:
def load_client(client_id):
    """Carga la partición privada de un cliente y escala los píxeles a [0, 1]."""
    with np.load(os.path.join(DATA_DIR, f"client_{client_id}.npz")) as data:
        x = np.expand_dims(data["x"] / 255.0, -1).astype("float32")
        y = data["y"]
    return x, y


_, (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_test = np.expand_dims(x_test / 255.0, -1).astype("float32")

### Ciclo de entrenamiento y evaluación

Para cada cliente $k$ y cada prefijo (`lmodel` o `lmodel_indep`) se guardan:
- `models/{prefijo}_{k}.keras`: el modelo local entrenado.
- `models/{prefijo}_{k}.json`: el número de ejemplos de entrenamiento $n_k$, que el servidor usa para ponderar el promedio.
- `reports/{prefijo}_{k}.txt`: el *classification report* sobre el conjunto de prueba.
- `viz/{prefijo}_{k}.png`: las curvas de aprendizaje (pérdida y exactitud).

In [ ]:
def plot_learning_curves(history, name):
    epochs = range(1, len(history.history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric, label in zip(axes, ("loss", "accuracy"), ("Pérdida", "Exactitud")):
        ax.plot(epochs, history.history[metric], marker="o", label="Entrenamiento")
        ax.plot(epochs, history.history[f"val_{metric}"], marker="o", label="Validación")
        ax.set_xticks(list(epochs))
        ax.set_xlabel("Época")
        ax.set_ylabel(label)
        ax.legend()
    fig.suptitle(f"Curvas de aprendizaje: {name}")
    fig.tight_layout()
    fig.savefig(f"viz/{name}.png")
    plt.show()


def train_and_evaluate(client_id, prefix, shared_init):
    name = f"{prefix}_{client_id}"
    x, y = load_client(client_id)
    x_train, x_val, y_train, y_val = train_test_split(
        x, y, test_size=VAL_SIZE, stratify=y, random_state=SEED
    )

    # Con w0 compartido, la misma semilla produce los mismos pesos iniciales en todos los clientes.
    tf.keras.utils.set_random_seed(SEED if shared_init else SEED + client_id + 1)
    model = build.build_it(name=name)

    history = model.fit(
        x_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(x_val, y_val),
        verbose=2,
    )

    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    report = classification_report(y_test, y_pred, digits=4)
    print(f"Classification report de {name} (conjunto de prueba):\n{report}")
    with open(f"reports/{name}.txt", "w", encoding="utf-8") as f:
        f.write(report)

    plot_learning_curves(history, name)

    model.save(f"models/{name}.keras")
    with open(f"models/{name}.json", "w", encoding="utf-8") as f:
        json.dump({"client_id": client_id, "n_samples": len(y_train)}, f)

In [ ]:
for prefix, shared_init in INIT_MODES.items():
    for client_id in CLIENT_IDS:
        train_and_evaluate(client_id, prefix, shared_init)